In [1]:
import pandas as pd

movies = pd.read_csv('../data/raw/movies.csv')
ratings = pd.read_csv('../data/raw/ratings.csv')

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

m = movies.copy()
m['genres'] = m['genres'].fillna('').str.replace('|', ' ', regex=False)

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(m['genres'])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

indices = pd.Series(m.index, index=m['title']).drop_duplicates()

def recommend(movie_title, top_n=10):
    if movie_title not in indices:
        return pd.Series([], dtype=str)
    idx = indices[movie_title]
    scores = list(enumerate(cosine_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    ids = [i[0] for i in scores]
    return m['title'].iloc[ids]

In [3]:
user_item = ratings.pivot_table(
    index='userId', columns='movieId', values='rating'
)

user_item_filled = user_item.fillna(0)

from sklearn.metrics.pairwise import cosine_similarity
user_sim = cosine_similarity(user_item_filled)

user_sim_df = pd.DataFrame(
    user_sim, index=user_item.index, columns=user_item.index
)

def recommend_cf(user_id, top_n=10):
    if user_id not in user_item.index:
        return pd.DataFrame(columns=['title'])
    
    sim_users = user_sim_df[user_id].sort_values(ascending=False).iloc[1:11]
    sim_users_movies = user_item.loc[sim_users.index]
    mean_ratings = sim_users_movies.mean().sort_values(ascending=False)
    
    seen = user_item.loc[user_id].dropna().index
    rec_ids = mean_ratings.drop(seen, errors='ignore').head(top_n).index
    
    return movies[movies['movieId'].isin(rec_ids)][['title']]

In [4]:
from sklearn.decomposition import TruncatedSVD
import numpy as np

ui = ratings.pivot_table(
    index='userId', columns='movieId', values='rating'
).fillna(0)

svd = TruncatedSVD(n_components=50, random_state=42)
latent = svd.fit_transform(ui)

recon = np.dot(latent, svd.components_)
recon_df = pd.DataFrame(recon, index=ui.index, columns=ui.columns)

def recommend_svd(user_id, top_n=10):
    if user_id not in recon_df.index:
        return pd.DataFrame(columns=['title'])
    
    user_scores = recon_df.loc[user_id]
    seen = ui.loc[user_id]
    seen = seen[seen > 0].index
    
    rec_ids = user_scores.drop(seen).sort_values(ascending=False).head(top_n).index
    return movies[movies['movieId'].isin(rec_ids)][['title']]

In [5]:
def hybrid_recommend(user_id, movie_title, top_n=5):
    content = recommend(movie_title, top_n*2)
    cf = recommend_cf(user_id, top_n*2)['title']
    svd = recommend_svd(user_id, top_n*2)['title']
    
    all_titles = list(content) + list(cf) + list(svd)
    
    counts = pd.Series(all_titles).value_counts()
    final = counts.head(top_n).index.tolist()
    
    return pd.DataFrame(final, columns=['title'])

In [6]:
hybrid_recommend(1, "Toy Story (1995)", top_n=5)

,title
0,Antz (1998)
1,Toy Story 2 (1999)
2,"Adventures of Rocky and Bullwinkle, The (2000)"
3,"Emperor's New Groove, The (2000)"
4,"Monsters, Inc. (2001)"
